# TEMPLATE

Runs part of the scenario-slice experiment (simrel) on Kaggle CPUs. Inputs: dataset `simrel-bundle` (code.zip, results.zip). Output: `out_TEMPLATE.zip` with every result file this run created or changed.


In [ ]:
TASK = 'TEMPLATE'
COMMANDS = ['...']
HOURS = 11.0  # anything still running then is stopped; finished jobs are already on disk and get packaged
import glob, os, shutil, subprocess, sys, time, zipfile
T0 = time.time()
def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found: attach the simrel-bundle dataset"
    return hits[0]
REPO = "/tmp/repo"
os.makedirs(REPO, exist_ok=True)
for part in ("code", "results"):
    hits = glob.glob(f"/kaggle/input/**/{part}.zip", recursive=True)
    if hits:
        zipfile.ZipFile(hits[0]).extractall(REPO)
    else:  # Kaggle may have unpacked the archive already
        src = os.path.dirname(find("slice_01_run.py" if part == "code" else "slices.json"))
        top = src[: src.rindex("/simrel/experiments")] if part == "code" else src[: src.rindex("/results/simrel/slices")]
        shutil.copytree(top, REPO, dirs_exist_ok=True)
print(os.listdir(REPO), os.path.exists(f"{REPO}/results/simrel/slices/slices.json"))


In [ ]:
os.chdir("/tmp")
def sh(c):
    print("$", c); subprocess.run(c, shell=True, check=True)
if not os.path.isdir("/tmp/abides"):
    sh("git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /tmp/abides && cd /tmp/abides && git checkout -q f9cbe51")
    sh(f"{sys.executable} -m pip install -q --no-deps -e /tmp/abides/abides-core -e /tmp/abides/abides-markets")
if not os.path.isdir("/tmp/lob_bench"):
    sh("git clone -q https://github.com/peernagy/lob_bench.git /tmp/lob_bench && cd /tmp/lob_bench && git checkout -q 276baf3")
sh(f"{sys.executable} -m pip install -q numba statsmodels sortedcontainers coloredlogs")
os.environ.update(LOB_BENCH_DIR="/tmp/lob_bench", OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", NUMBA_NUM_THREADS="1")
sh(f"cd {REPO} && {sys.executable} -c 'import sys; sys.path.insert(0, \".\"); import simbench.abides_world, simrel.metrics.lobbench as lb, simrel.adapters.hawkes_fast; lb._modules(); print(\"imports ok\")'")
MARK = time.time()


In [ ]:
for c in COMMANDS:
    left = HOURS * 3600 - (time.time() - T0)
    if left < 600:
        print("no time left for", c); break
    print("==", c, f"({left / 3600:.1f} h left)", flush=True)
    p = subprocess.Popen(f"cd {REPO} && {sys.executable} -u simrel/experiments/{c}", shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in p.stdout:
            if "Warning" not in line and "to_timedelta" not in line:
                print(line, end="", flush=True)
            if time.time() - T0 > HOURS * 3600:
                p.kill(); print("stopped at the time limit"); break
    finally:
        p.wait()


In [ ]:
# package every result file created or changed by this run
out = "/kaggle/working/out_TEMPLATE.zip"
n = 0
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(f"{REPO}/results/simrel/slices"):
        for f in files:
            p = os.path.join(root, f)
            if os.path.getmtime(p) >= MARK - 5:
                z.write(p, os.path.relpath(p, REPO)); n += 1
print(n, "files ->", out, round(os.path.getsize(out) / 1e6), "MB")
